# Constrained optimization, Lagrange multipliers, and KKT

KKT says the objective gradient at a constrained optimum is exactly balanced by active constraint gradients.

Constraints add walls to unconstrained optimality. Convexity makes KKT sufficient, and duality grows out of the multipliers introduced here. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 271828
rng = np.random.default_rng(SEED)


def sigmoid(z):
    clipped = np.clip(z, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-clipped))


def soft_threshold(v, threshold):
    return np.sign(v) * np.maximum(np.abs(v) - threshold, 0.0)


def quadratic_loss(A, b, x):
    return 0.5 * float(x @ A @ x) - float(b @ x)


def quadratic_grad(A, b, x):
    return A @ x - b


def rosenbrock_loss(x):
    a = 1.0 - x[0]
    b = x[1] - x[0] ** 2
    ripple = 0.08 * np.sin(3.0 * x[0]) * np.cos(2.0 * x[1])
    return a ** 2 + 35.0 * b ** 2 + ripple


def rosenbrock_grad(x):
    dx = -2.0 * (1.0 - x[0]) - 140.0 * x[0] * (x[1] - x[0] ** 2)
    dy = 70.0 * (x[1] - x[0] ** 2)
    dx = dx + 0.24 * np.cos(3.0 * x[0]) * np.cos(2.0 * x[1])
    dy = dy - 0.16 * np.sin(3.0 * x[0]) * np.sin(2.0 * x[1])
    return np.array([dx, dy])


def make_logistic_data(n=96, d=2, seed=11):
    local = np.random.default_rng(seed)
    half = n // 2
    pos = local.normal(loc=1.15, scale=0.55, size=(half, d))
    neg = local.normal(loc=-1.05, scale=0.65, size=(n - half, d))
    X = np.vstack([pos, neg])
    y = np.hstack([np.ones(half), -np.ones(n - half)])
    return X, y


def make_sparse_logistic_data(n=140, d=32, seed=23):
    local = np.random.default_rng(seed)
    X = local.normal(size=(n, d))
    mask = local.random(size=X.shape) < 0.72
    X[mask] = 0.0
    true_w = np.zeros(d)
    true_w[:5] = np.array([1.4, -1.1, 0.9, -0.7, 0.45])
    logits = X @ true_w + 0.15 * local.normal(size=n)
    y = np.where(logits >= 0.0, 1.0, -1.0)
    return X, y


def logistic_loss(w, X, y, lam=0.0):
    margins = y * (X @ w)
    data = np.logaddexp(0.0, -margins).mean()
    penalty = 0.5 * lam * float(w @ w)
    return float(data + penalty)


def logistic_grad(w, X, y, lam=0.0):
    margins = y * (X @ w)
    weights = -y * sigmoid(-margins)
    grad = X.T @ weights / X.shape[0]
    return grad + lam * w


def l1_logistic_objective(w, X, y, lam=0.04):
    return logistic_loss(w, X, y, 0.0) + lam * float(np.abs(w).sum())


def project_box(x, lo=-2.0, hi=2.0):
    return np.clip(x, lo, hi)


def project_l2_ball(x, radius=2.0):
    norm = np.linalg.norm(x)
    if norm <= radius:
        return x.copy()
    return x * (radius / norm)


def make_loss_surface_ladder():
    X2, y2 = make_logistic_data()
    Xh, yh = make_sparse_logistic_data()
    d = Xh.shape[1]
    A1 = np.array([[4.0, 1.0], [1.0, 3.0]])
    b1 = np.array([1.0, 2.0])
    A2 = np.array([[45.0, 18.0], [18.0, 9.0]])
    b2 = np.array([1.0, 0.4])
    return [
        {
            "id": "D1",
            "name": "quadratic bowl",
            "x0": np.array([0.0, 0.0]),
            "loss": lambda x, A=A1, b=b1: quadratic_loss(A, b, x),
            "grad": lambda x, A=A1, b=b1: quadratic_grad(A, b, x),
            "project": lambda x: x,
            "dim": 2,
            "info": "2-D SPD quadratic with closed-form minimizer",
        },
        {
            "id": "D2",
            "name": "ill-conditioned quadratic",
            "x0": np.array([1.8, -1.5]),
            "loss": lambda x, A=A2, b=b2: quadratic_loss(A, b, x),
            "grad": lambda x, A=A2, b=b2: quadratic_grad(A, b, x),
            "project": lambda x: x,
            "dim": 2,
            "info": "anisotropic bowl with coupled coordinates",
        },
        {
            "id": "D3",
            "name": "nonconvex Rosenbrock-ripple",
            "x0": np.array([-1.2, 1.0]),
            "loss": rosenbrock_loss,
            "grad": rosenbrock_grad,
            "project": lambda x: x,
            "dim": 2,
            "info": "curved valley plus small multimodal ripple",
        },
        {
            "id": "D4",
            "name": "real logistic loss",
            "x0": np.zeros(2),
            "loss": lambda w, X=X2, y=y2: logistic_loss(w, X, y, 0.02),
            "grad": lambda w, X=X2, y=y2: logistic_grad(w, X, y, 0.02),
            "project": lambda x: x,
            "dim": 2,
            "X": X2,
            "y": y2,
            "info": "NumPy logistic regression objective on a fixed small dataset",
        },
        {
            "id": "D5",
            "name": "high-dimensional sparse constrained case",
            "x0": np.zeros(d),
            "loss": lambda w, X=Xh, y=yh: l1_logistic_objective(w, X, y, 0.04),
            "grad": lambda w, X=Xh, y=yh: logistic_grad(w, X, y, 0.0),
            "project": lambda x: project_l2_ball(x, 2.5),
            "dim": d,
            "X": Xh,
            "y": yh,
            "A": A5,
            "b": b5,
            "info": "32-D sparse logistic objective with L1 and norm constraint",
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        sample = rung["x0"][: min(5, rung["dim"])]
        print(rung["id"], rung["name"], "dim=", rung["dim"], "sample=", np.round(sample, 3), "--", rung["info"])


def contour_values(rung, grid=80, span=2.4):
    xs = np.linspace(-span, span, grid)
    ys = np.linspace(-span, span, grid)
    Z = np.zeros((grid, grid))
    base = rung["x0"].astype(float).copy()
    for i, xval in enumerate(xs):
        for j, yval in enumerate(ys):
            probe = base.copy()
            probe[0] = xval
            probe[1] = yval
            Z[j, i] = rung["loss"](probe)
    return xs, ys, Z


def plot_trajectory_summary(results, metric_label="final loss"):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for col, item in enumerate(results):
        rung = item["rung"]
        path = item["path"]
        losses = item["losses"]
        xs, ys, Z = contour_values(rung)
        axes[0, col].contour(xs, ys, Z, levels=18, cmap="viridis")
        axes[0, col].plot(path[:, 0], path[:, 1], marker="o", markersize=2, linewidth=1)
        axes[0, col].set_title(rung["id"])
        axes[1, col].plot(losses)
        axes[1, col].set_title(metric_label)
        axes[1, col].set_xlabel("iteration")
    plt.tight_layout()


def print_metric_table(results, metric_label="final loss"):
    print(f"{'rung':<4} {'name':<38} {'iters':>6} {metric_label:>14}")
    for item in results:
        final_value = item["metric"]
        iters = len(item["losses"]) - 1
        print(f"{item['rung']['id']:<4} {item['rung']['name']:<38} {iters:>6} {final_value:>14.6f}")

## The concept, built once (D1)

KKT combines stationarity $\nabla f+\sum_i\mu_i\nabla g_i+\sum_j\lambda_j\nabla h_j=0$, feasibility, $\mu_i\ge0$, and $\mu_i g_i(x)=0$.

Verify the equality-constrained quadratic and inactive inequality arithmetic from the lesson. The multiplier balances objective force against the wall normal.

In [ ]:
def kkt_equality_solution():
    x = np.array([0.5, 0.5])
    grad_f = 2.0 * x
    grad_h = np.array([1.0, 1.0])
    lam = -1.0
    stationarity = grad_f + lam * grad_h
    return x, lam, stationarity


def inactive_complementarity(x, mu):
    g_value = 1.0 - x
    return g_value, mu * g_value


x_star, lam_star, stationarity = kkt_equality_solution()
g_inactive, comp = inactive_complementarity(2.0, 0.0)

assert np.allclose(x_star, np.array([0.5, 0.5]))
assert np.isclose(lam_star, -1.0)
assert np.allclose(stationarity, np.array([0.0, 0.0]))
assert np.isclose(g_inactive, -1.0)
assert np.isclose(comp, 0.0)
print(x_star, lam_star, stationarity)
print("inactive g", g_inactive, "mu g", comp)

Build a projected optimizer with KKT diagnostics. Projection maintains primal feasibility; the diagnostic checks stationarity residuals and complementarity on simple constraints.

In [ ]:
def projected_optimizer_with_kkt_check(rung, steps=100, eta0=0.12):
    x = rung["x0"].astype(float).copy()
    path = [x.copy()]
    losses = [rung["loss"](x)]
    residuals = []
    for t in range(steps):
        eta = eta0 / math.sqrt(t + 1.0)
        grad = rung["grad"](x)
        x = rung["project"](x - eta * grad)
        residuals.append(np.linalg.norm(rung["grad"](x)))
        path.append(x.copy())
        losses.append(rung["loss"](x))
    return np.array(path), np.array(losses), np.array(residuals)

## The dataset ladder

Family F4 uses the same inline D1-D5 ladder: quadratic, ill-conditioned, nonconvex, real logistic loss, and high-dimensional sparse constrained loss.

In [ ]:
ladder = make_loss_surface_ladder()
preview_ladder(ladder)

## Run the same method across D1-D5

The metric is the plan's requested final loss, final objective, feasible loss, or primal-dual gap.

In [ ]:
ladder = make_loss_surface_ladder()
for rung in ladder:
    rung["project"] = lambda x, project=rung["project"]: project_box(project(x), -2.0, 2.0)
results = []
for rung in ladder:
    path, losses, residuals = projected_optimizer_with_kkt_check(rung, steps=120, eta0=0.11)
    results.append({"rung": rung, "path": path, "losses": losses, "residuals": residuals, "metric": losses[-1]})
print_metric_table(results, "feasible loss")

## Results visualization

The closing figure has trajectory-on-contours panels and a summary metric curve.

In [ ]:
plot_trajectory_summary(results, "feasible loss")
metrics = [item["metric"] for item in results]
plt.figure(figsize=(6, 3))
plt.plot([item["rung"]["id"] for item in results], metrics, marker="o")
plt.ylabel("final feasible loss")
plt.title("Projected optimizer by rung")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on the hardest rung

Pitfall on D5: forcing inactive inequalities to equality or accepting negative multipliers. The fix is to enforce complementarity and dual feasibility separately.

In [ ]:
def bad_force_every_wall_active(x):
    g_values = np.array([x[0] - 2.0, -x[0] - 2.0])
    mu = np.array([1.0, -0.5])
    return g_values, mu, mu * g_values


def fixed_kkt_wall_check(x):
    g_values = np.array([x[0] - 2.0, -x[0] - 2.0])
    mu = np.maximum(np.array([1.0, -0.5]), 0.0)
    inactive = g_values < -1e-8
    mu[inactive] = 0.0
    return g_values, mu, mu * g_values


hard_point = results[-1]["path"][-1]
wrong_g, wrong_mu, wrong_comp = bad_force_every_wall_active(hard_point)
fixed_g, fixed_mu, fixed_comp = fixed_kkt_wall_check(hard_point)
print("wrong mu", wrong_mu, "wrong mu*g", wrong_comp)
print("fixed mu", fixed_mu, "fixed mu*g", fixed_comp)

## Evaluate it + Practice

- Metric: compare the final value to a no-skill baseline that returns the initial point.
- Sanity check: on D1, verify the path moves toward the closed-form quadratic solution or the asserted lesson numbers.
- Ablation: turn off the key idea, such as newest-value updates, the prox, projection, dual lower bound, uniform sampling, or PSD check.
- Failure signals: rising loss, exploding iterates, negative inequality multipliers, invalid lower bounds, or dense non-sparse D5 solutions.
- CPU note: these examples are seeded, small, and NumPy-only; do not execute the notebook as part of rebuilding.

Practice: Replace the box with a simplex projection and inspect active coordinates.

Practice: Compute a finite-difference stationarity residual after projection on D2.

Practice: Construct a degenerate pair of duplicate constraints and observe nonunique multipliers.